> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataCollection/blob/main/notebooks/01c_Joining_Tables_to_Maps.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Joining Census Tables to Maps

**Series:** Census Data in Python (notebook 3 of 3)
**Time:** about 35 minutes

**In this notebook you will:**
- Join ACS data from the API to census tract boundaries using GEOID
- Find and fix rows that do not match
- Map the share of households without a car in Tippecanoe County
- Show where estimates are unreliable

**You should already know:** Notebooks 1a and 1b. You also need a free Census API key.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## The question

Which neighborhoods in Tippecanoe County have the most households **without a car**? These are the
places where the distance to a grocery store matters most, which is the theme of the grocery access
case study in the Case Studies course.

The data comes in two pieces that we have to put together:

1. **Shapes**: census tract boundaries (notebook 1a)
2. **Numbers**: ACS estimates from the Census API (notebook 1b)

The key that links them is the tract **GEOID**.

## Your Census API key

> **Every learner needs their own free Census API key.** Request one at
> <https://api.census.gov/data/key_signup.html>. It arrives by email within a few minutes, and you
> must click the activation link in that email before it works.

Paste your key between the quotes in the next cell and run it. Treat the key like a password: do not
post it online or leave it in a notebook you share. If you share this notebook, delete the key first.

In [ ]:
CENSUS_API_KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"

if CENSUS_API_KEY.startswith("PASTE"):
    raise ValueError("Paste your free Census API key between the quotes above, then run this cell again.")
print("Key saved. It is", len(CENSUS_API_KEY), "characters long (a valid key has 40).")

## Step 1: Import libraries and the helper function from notebook 1b

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

def census_get(dataset, variables, geo_for, geo_in=None):
    """Ask the Census API for some variables and return a pandas DataFrame.

    dataset   : the part of the address after /data/, for example "2024/acs/acs5"
    variables : a list of variable codes, for example ["NAME", "B01003_001E"]
    geo_for   : the geography you want, for example "county:*" (all counties)
    geo_in    : the geography it sits inside, for example "state:18" (Indiana)
    """
    params = {"get": ",".join(variables), "for": geo_for, "key": CENSUS_API_KEY}
    if geo_in:
        params["in"] = geo_in
    reply = requests.get(f"https://api.census.gov/data/{dataset}", params=params, timeout=60)

    # The API answers with JSON when it works, and with a web page or a short message when it does not
    if reply.status_code != 200 or "json" not in reply.headers.get("Content-Type", ""):
        raise RuntimeError(
            f"The Census API did not return data (status {reply.status_code}). "
            f"The start of its reply was:\n{reply.text[:300]}"
        )

    rows = reply.json()                                  # a list of lists; the first list is the header
    table = pd.DataFrame(rows[1:], columns=rows[0])
    for v in variables:                                  # the API sends every value as text
        if v != "NAME":
            table[v] = pd.to_numeric(table[v], errors="coerce")
    return table

## Step 2: The shapes

In [ ]:
tracts = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_18_tract_500k.zip")
tracts = tracts[tracts["COUNTYFP"] == "157"][["GEOID", "NAMELSAD", "geometry"]]
print(len(tracts), "tract shapes")
tracts.head(3)

## Step 3: The numbers

Table **B08201** (Household Size by Vehicles Available) gives the total number of households
(`B08201_001E`) and the number with no vehicle (`B08201_002E`), each with a margin of error.

In [ ]:
acs = census_get(
    "2024/acs/acs5",
    ["NAME", "B08201_001E", "B08201_001M", "B08201_002E", "B08201_002M"],
    "tract:*", "state:18 county:157",
)
acs = acs.rename(columns={
    "B08201_001E": "households", "B08201_001M": "households_moe",
    "B08201_002E": "no_car", "B08201_002M": "no_car_moe",
})
print(len(acs), "rows from the API")
acs.head(3)

## Step 4: Build the GEOID

The API returns the state, county and tract codes in separate columns. Joined together they make the
same 11-character GEOID that the shapes have.

In [ ]:
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]
acs[["NAME", "state", "county", "tract", "GEOID"]].head(3)

### A common mistake: numbers instead of text

If a GEOID is stored as a number, any leading zero disappears and the join silently fails. Indiana's
code (18) has no leading zero, but many states do. Here is what happens to Alabama's first county:

In [ ]:
geoid_text = "01001"
geoid_number = int(geoid_text)
print("As text:  ", geoid_text)
print("As number:", geoid_number, " <- the leading zero is gone, so it no longer matches")
print("Fixed:    ", str(geoid_number).zfill(5))

## Step 5: Join, and check what did not match

`merge` joins two tables on a shared column. `indicator=True` adds a `_merge` column that tells us
whether each row was found in both tables, or only one. Always check this.

In [ ]:
check = tracts.merge(acs, on="GEOID", how="outer", indicator=True)
check["_merge"].value_counts()

Every row says `both`, so every shape found its numbers. If you saw `left_only` or `right_only`, the
usual causes are a GEOID stored as a number, boundaries and data from different years (tracts are
redrawn after each decennial census), or a filter applied to one table but not the other.

Now we do the real join. Starting from the shapes keeps the result a GeoDataFrame.

In [ ]:
tipp = tracts.merge(acs, on="GEOID", how="left")
type(tipp)

## Step 6: Calculate the percentage

Some tracts have **no households**, because almost everyone there lives in university residence halls
(group quarters). Dividing by zero would give a meaningless result, so we set those to missing.

In [ ]:
tipp["pct_no_car"] = np.where(tipp["households"] > 0,
                              100 * tipp["no_car"] / tipp["households"],
                              np.nan)
tipp[["NAMELSAD", "households", "no_car", "pct_no_car"]].sort_values("pct_no_car", ascending=False).head()

## Step 7: Map it

We use a projected coordinate system for Indiana (UTM zone 16N, EPSG:26916), five quantile classes, a
sequential color scheme, and gray for tracts with no households.

In [ ]:
tipp = tipp.to_crs(epsg=26916)

ax = tipp.plot(
    column="pct_no_car", scheme="Quantiles", k=5, cmap="YlOrRd",
    edgecolor="white", linewidth=0.5, figsize=(8, 8), legend=True,
    legend_kwds={"title": "Households without a car (%)", "loc": "lower left", "fmt": "{:.0f}"},
    missing_kwds={"color": "lightgray", "label": "No households"},
)
ax.set_title("Households without a vehicle, Tippecanoe County, Indiana")
ax.annotate("Source: U.S. Census Bureau, ACS 5-year estimates 2020-2024, table B08201",
            xy=(0.01, -0.02), xycoords="axes fraction", fontsize=8, color="gray")
ax.set_axis_off()
plt.show()

The highest shares are in the central tracts of Lafayette and West Lafayette, close to downtown and the
Purdue campus, where many residents are students or live in apartments near shops and bus routes.

## Step 8: How reliable is each estimate?

Tract estimates come from small samples, so their margins of error can be large. We calculate the
coefficient of variation (CV) of the no-car count (see notebook 1b) and outline the tracts where the
estimate has low reliability (CV above 40%).

In [ ]:
tipp["cv"] = np.where(tipp["no_car"] > 0, (tipp["no_car_moe"] / 1.645) / tipp["no_car"] * 100, np.nan)
unreliable = tipp[tipp["cv"] > 40]
print(len(unreliable), "of", len(tipp), "tracts have a low-reliability estimate")

ax = tipp.plot(
    column="pct_no_car", scheme="Quantiles", k=5, cmap="YlOrRd",
    edgecolor="white", linewidth=0.5, figsize=(8, 8), legend=True,
    legend_kwds={"title": "Households without a car (%)", "loc": "lower left", "fmt": "{:.0f}"},
    missing_kwds={"color": "lightgray", "label": "No households"},
)
unreliable.plot(ax=ax, facecolor="none", edgecolor="black", hatch="///", linewidth=0.8)
ax.set_title("Hatched tracts: estimate has low reliability (CV above 40%)")
ax.set_axis_off()
plt.show()

Showing uncertainty on the map is honest: readers can see which differences are solid and which might
just be sampling noise.

## Step 9: An interactive map

In [ ]:
tipp.explore(
    column="pct_no_car", scheme="Quantiles", k=5, cmap="YlOrRd",
    tooltip=["NAMELSAD", "households", "no_car", "pct_no_car", "cv"],
    tiles="Esri.WorldGrayCanvas", style_kwds={"weight": 1, "color": "white"},
    missing_kwds={"color": "lightgray"},
)

## Try it yourself

Make a county map of Indiana showing the **percent of people aged 65 and over** in 2020.

1. Get `NAME` and `DP1_0024P` for all Indiana counties from the dataset `2020/dec/dp`.
2. Build a 5-character county GEOID from the `state` and `county` columns.
3. Read the county boundaries from
   `https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_5m.zip` and keep Indiana (`STATEFP` 18).
4. Join on `GEOID` and map the percentage.

*Hint:* Steps 4 and 5 of this notebook show how to build the GEOID and join.

<details>
<summary><b>Show a solution</b></summary>

```python
older = census_get("2020/dec/dp", ["NAME", "DP1_0024P"], "county:*", "state:18")
older["GEOID"] = older["state"] + older["county"]

counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_5m.zip")
counties = counties[counties["STATEFP"] == "18"]

in_older = counties.merge(older, on="GEOID")
ax = in_older.plot(column="DP1_0024P", cmap="Blues", scheme="Quantiles", k=5, legend=True,
                   edgecolor="white", figsize=(6, 8),
                   legend_kwds={"title": "Age 65 and over (%)", "loc": "lower left"})
ax.set_title("Population aged 65 and over, Indiana, 2020")
ax.set_axis_off()
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- Census numbers and census shapes are joined on GEOID, which must be text
- Use `indicator=True` to check that every row matched
- Boundaries and data should come from compatible years
- Guard against dividing by zero, and show missing data in a neutral color
- Mark unreliable estimates so readers do not over-read them